# Phase 3 — Leakage-free multi-task model on CBIS-DDSM

**Question:** once the split is honest, does the paper's design (one network, two task heads, CBAM attention) actually help?

Four variants, trained on the **same patient-level folds** (no patient in both train and test) so they can be compared image by image:

| Variant | Malignancy head | Density head | CBAM attention |
|---|:-:|:-:|:-:|
| `mt_cbam` (paper design) | ✓ | ✓ | ✓ |
| `mt_plain` | ✓ | ✓ | – |
| `st_path` | ✓ | – | ✓ |
| `st_dens` | – | ✓ | ✓ |

**Before running**
1. Settings → Accelerator → **GPU T4 x2**. Settings → **Internet on**.
2. Add Input → search **"CBIS-DDSM: Breast Cancer Image Dataset"** (by *awsaf49*) → Add.
3. Run the first **two** cells (about 5 minutes). If cell 2 ends with `ALL CHECKS PASSED`, click **Save Version → Save & Run All (Commit)**. The full run takes about **2 hours** and continues if you close the browser.
4. When it finishes, open the saved version → **Output** → download **`cbis_results.zip`** and send it to Claude.

In [ ]:
# 1) Get the code and check the machine
import os, subprocess, sys, torch
REPO = "/kaggle/working/repo"
if not os.path.exists(REPO):
    r = subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sweetmAGIciaN7/mammography-multitask-ai.git", REPO],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("Could not download the code. Is Internet ON (Settings -> Internet)?\n" + r.stderr)
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)
N_GPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPU)] or "NONE")
assert N_GPU > 0, "Turn on the GPU: Settings -> Accelerator -> GPU T4 x2, then run again."
os.environ["PYTHONPATH"] = f"{REPO}/src"

In [ ]:
# 2) Unit tests + 3-minute smoke run on 40 patients. If this cell fails, stop and send Claude a screenshot.
!cd /kaggle/working/repo && python -m pytest -q tests 2>&1 | tail -3
!cd /kaggle/working/repo && python -m mammo.experiments.cbis prepare --smoke \
  && python -m mammo.experiments.cbis run --smoke --official \
  && python -m mammo.experiments.cbis summarise --smoke \
  && echo "ALL CHECKS PASSED"

In [ ]:
# 3) Index CBIS-DDSM and preprocess every mammogram once (~5-10 min). Check the preview: breasts should
#    face left, fill the frame, and have no big white film borders.
!cd /kaggle/working/repo && python -m mammo.experiments.cbis prepare
from IPython.display import Image, display
display(Image("/kaggle/working/results/cbis/preprocessing_preview.png"))

In [ ]:
# 4) Train all variants x 5 folds (+ the main model on the official split), ~2 h on T4 x2.
#    With two GPUs, two variants train at the same time. Progress is printed every 2 minutes.
import subprocess, time
plan = {0: ["mt_cbam", "st_dens"], 1: ["mt_plain", "st_path"]} if N_GPU >= 2 else {0: ["mt_cbam", "st_dens", "mt_plain", "st_path"]}
procs = {}
for gpu, variants in plan.items():
    log = open(f"/kaggle/working/train_gpu{gpu}.log", "w")
    cmd = [sys.executable, "-m", "mammo.experiments.cbis", "run", *variants, "--official", "mt_cbam", "--gpu", str(gpu)]
    procs[gpu] = subprocess.Popen(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())
    print(f"GPU {gpu}: {variants}")

def last_line(gpu):
    lines = [l for l in open(f"/kaggle/working/train_gpu{gpu}.log").read().splitlines() if l.strip()]
    return lines[-1] if lines else "(starting)"

t0 = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(120)
    print(f"[{(time.time() - t0) / 60:5.0f} min] " + " | ".join(f"GPU {g}: {last_line(g)}" for g in procs), flush=True)
for gpu, p in procs.items():
    if p.returncode != 0:
        print(open(f"/kaggle/working/train_gpu{gpu}.log").read()[-5000:])
        raise SystemExit(f"GPU {gpu} failed (see above). Send Claude this output.")
print(f"training finished in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# 5) Results: tables, paired comparisons and the chart
!cd /kaggle/working/repo && python -m mammo.experiments.cbis summarise
display(Image("/kaggle/working/results/cbis/ablation_chart.png"))

In [ ]:
# 6) Pack everything for Claude -> download cbis_results.zip from Output (/kaggle/working)
!cp /kaggle/working/train_gpu*.log /kaggle/working/results/cbis/ 2>/dev/null; cd /kaggle/working && rm -f cbis_results.zip && zip -qr cbis_results.zip results/cbis && ls -lh cbis_results.zip checkpoints/